# 🛒 Online Retail II – Customer Segmentation, Repeat-Purchase Prediction & Spend Forecast

## 🎯 Business Problem
An online gift-ware shop sells to thousands of customers. The owner wants three answers:
1. **Who are my customer groups?** (VIP, regular, low value ...) → *Clustering*
2. **Which customers will buy again in the next 90 days?** → *Classification*
3. **How much will a returning customer spend in the next 90 days?** → *Regression*

With these answers the shop can send offers to the right people instead of everyone.

## 📦 Dataset
**Online Retail II** (UK online store, Dec 2009 – Dec 2011). Columns: `Invoice, StockCode, Description, Quantity, InvoiceDate, Price, Customer ID, Country`.

## 🧠 Models used
Logistic Regression, KNN, Gaussian Naive Bayes, Decision Tree, Random Forest, Linear Regression, K-Means, Hierarchical Clustering.

## 🧭 Plan of the notebook
Load → Clean → EDA (chart 1) → Feature engineering → Classification (chart 2) → Regression → Clustering (chart 3) → Conclusion

## 1. Import Important Libraries
pandas + numpy for data work, seaborn + matplotlib for the 3 charts, sklearn for models and scores.

In [ ]:
import pandas as pd  # pandas: table (DataFrame) banane aur data saaf karne ke liye
import numpy as np  # numpy: array par tez maths ke liye
import matplotlib.pyplot as plt  # matplotlib: basic charts ke liye
import seaborn as sns  # seaborn: matplotlib ke upar sundar charts ke liye
import warnings  # warnings: bekaar warning messages chhupane ke liye
warnings.filterwarnings("ignore")  # warnings band kar di taaki output saaf dikhe

In [ ]:
from sklearn.model_selection import train_test_split  # data ko train aur test hisson mein baantne ke liye
from sklearn.preprocessing import StandardScaler  # saare columns ko same scale par lane ke liye
from sklearn.linear_model import LogisticRegression, LinearRegression  # linear models
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor  # KNN models
from sklearn.naive_bayes import GaussianNB  # Gaussian Naive Bayes classifier
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor  # decision tree models
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor  # random forest models
from sklearn.cluster import KMeans, AgglomerativeClustering  # K-Means aur hierarchical clustering

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix  # classification ke scores
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, r2_score  # regression ke error scores
from sklearn.metrics import silhouette_score, adjusted_rand_score  # clustering ki quality ke scores

## 2. Load the Data
We read the CSV file `online_retail_II.csv` with `pd.read_csv` into a pandas DataFrame.
Keep the file in the same folder as this notebook.

In [ ]:
df_raw = pd.read_csv("online_retail_II.csv")  # csv file padhi
df_raw.shape  # rows aur columns ki ginti

In [ ]:
df_raw.head()  # pehli 5 rows dekhne ke liye

In [ ]:
df_raw.info()  # har column ka data type aur non-null count

## 3. Data Quality Check
Before cleaning we must **see the problems**: missing customers, cancelled invoices, returns and duplicates.

In [ ]:
df_raw.isnull().sum()  # har column mein missing values ki ginti

In [ ]:
print(df_raw.duplicated().sum())  # bilkul same (duplicate) rows ki ginti
print((df_raw["Quantity"] <= 0).sum())  # zero ya negative quantity wali rows (returns)
print(df_raw["Invoice"].astype(str).str.startswith("C").sum())  # cancel hue invoices (ye C letter se shuru hote hain)

**What we found:** some rows have no `Customer ID`, some invoices are cancellations (negative quantity) and some rows are duplicates.
We cannot track a customer without an ID, so those rows must go. Cancelled invoices are not real sales, so they go too.

## 4. Data Cleaning
One small function does all the cleaning, so it is easy to explain step by step.

In [ ]:
def clean_data(df):  # function jo raw data ko saaf karta hai
    df = df.dropna(subset=["Customer ID"]).copy()  # jin rows mein customer id nahi hai unhe hata do (customer track nahi ho sakta)
    df["Invoice"] = df["Invoice"].astype(str)  # invoice number ko text banaya taaki C letter check kar sakein
    df = df[~df["Invoice"].str.startswith("C")]  # cancel hue invoices hata diye
    df = df[(df["Quantity"] > 0) & (df["Price"] > 0)]  # sirf asli sales rakhi jinmein quantity aur price positive hai
    df = df[df["StockCode"] != "POST"]  # postage charges hata diye (ye asli product nahi hai)
    df = df.drop_duplicates()  # bilkul same duplicate rows hata di
    return df  # saaf kiya hua table wapas bhej diya

In [ ]:
def fix_types(df):  # function jo data types theek karta hai aur Revenue column jodta hai
    df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], dayfirst=True)  # text ko asli date mein badla (pehle din aata hai)
    df["Customer ID"] = df["Customer ID"].astype(int)  # customer id ko poora number (integer) banaya
    df["Revenue"] = df["Quantity"] * df["Price"]  # har row par kamaya hua paisa
    return df  # table wapas bhej diya

In [ ]:
df = clean_data(df_raw)  # step 1: data saaf karo
df = fix_types(df)  # step 2: types theek karo
print(df_raw.shape, "->", df.shape)  # cleaning se pehle aur baad ki rows
print(df["Customer ID"].nunique())  # unique customers ki ginti
print(df["InvoiceDate"].min(), df["InvoiceDate"].max())  # data ki pehli aur aakhri date

## 5. Exploratory Data Analysis (EDA)
### 📈 Chart 1 – Monthly Revenue
Shows how the business grows and when the **seasonal peak** happens (gift shops sell most before Christmas).
The last month can look low because the data stops in the middle of December.

In [ ]:
def plot_monthly_revenue(df):  # function jo monthly revenue ka chart banata hai
    monthly = df.groupby(df["InvoiceDate"].dt.to_period("M"))["Revenue"].sum()  # har mahine ka total revenue
    monthly.plot(figsize=(10, 4), marker="o")  # dots ke saath line chart banaya
    plt.title("Monthly Revenue (£)")  # chart ka title
    plt.xlabel("Month")  # x axis ka label
    plt.ylabel("Revenue (£)")  # y axis ka label
    plt.show()  # chart dikhao

In [ ]:
plot_monthly_revenue(df)  # function call karke chart 1 banao

### 💡 Two quick business facts (numbers instead of more charts)
The **Pareto rule** (80/20) – do a few customers give most of the money? And which countries matter?

In [ ]:
cust_rev = df.groupby("Customer ID")["Revenue"].sum().sort_values(ascending=False)  # har customer ka total kharch, sabse zyada upar
top20 = cust_rev.head(int(len(cust_rev) * 0.2)).sum() / cust_rev.sum()  # top 20% customers se aane wale revenue ka hissa
print(f"Top 20% customers give {top20:.1%} of the revenue")  # Pareto ka result print karo

In [ ]:
country_rev = df.groupby("Country")["Revenue"].sum().sort_values(ascending=False)  # har country ka revenue, sabse zyada upar
(country_rev.head(5) / country_rev.sum()).round(3)  # top 5 countries ka revenue hissa

## 6. Feature Engineering – Build One Row Per Customer
The raw data has one row per product line. For machine learning we need **one row per customer**.

⚠️ **Avoiding data leakage (very important for interviews):**
We pick a **cutoff date = 90 days before the data ends**.
- Features (inputs) are made **only from purchases BEFORE the cutoff** (the past).
- Targets (what we predict) are made **only from purchases AFTER the cutoff** (the future).

So the model never sees the future while learning – exactly like real life.

In [ ]:
cutoff = df["InvoiceDate"].max() - pd.Timedelta(days=90)  # hamara "aaj": data khatam hone se 90 din pehle
past = df[df["InvoiceDate"] < cutoff]  # cutoff se pehle ki purchases (features ke liye)
future = df[df["InvoiceDate"] >= cutoff]  # cutoff ke baad ki purchases (target ke liye)
print(cutoff)  # cutoff date dikhao

### RFM features
- **Recency** – days since the last purchase (small = active customer)
- **Frequency** – number of orders
- **Monetary** – total money spent

We also add a few more easy features: total items, unique products, customer age, average order value, spend in the last 90 days before the cutoff, and a UK flag.

In [ ]:
def make_features(past, cutoff):  # function jo past data se har customer ki ek row banata hai
    g = past.groupby("Customer ID")  # past purchases ko customer ke hisaab se group kiya
    feat = g["Revenue"].sum().to_frame("Monetary")  # Monetary: har customer ne kul kitna paisa kharch kiya
    feat["Frequency"] = g["Invoice"].nunique()  # Frequency: orders ki ginti
    feat["Recency"] = (cutoff - g["InvoiceDate"].max()).dt.days  # Recency: aakhri purchase ko kitne din hue
    feat["TotalItems"] = g["Quantity"].sum()  # kul kitne items kharide
    feat["UniqueProducts"] = g["StockCode"].nunique()  # kitne alag-alag products kharide
    feat["CustomerAgeDays"] = (cutoff - g["InvoiceDate"].min()).dt.days  # pehli purchase ko kitne din hue
    feat["AvgOrderValue"] = feat["Monetary"] / feat["Frequency"]  # ek order ka average paisa
    return feat  # feature table wapas bhej diya

In [ ]:
feat = make_features(past, cutoff)  # customer table banao
recent = past[past["InvoiceDate"] >= cutoff - pd.Timedelta(days=90)]  # cutoff se pehle ke aakhri 90 din ki purchases
feat["Spend_Last90"] = recent.groupby("Customer ID")["Revenue"].sum()  # un 90 dinon ka kharch (customer id se match hua)
feat["Spend_Last90"] = feat["Spend_Last90"].fillna(0)  # jisne recent purchase nahi ki use 0 mila
feat["IsUK"] = (past.groupby("Customer ID")["Country"].first() == "United Kingdom").astype(int)  # 1 agar customer UK ka hai

### Targets (what we want to predict) – from the FUTURE data
- `FutureSpend` – money spent in the next 90 days
- `WillBuy` – 1 if the customer bought again, else 0

In [ ]:
feat["FutureSpend"] = future.groupby("Customer ID")["Revenue"].sum()  # har customer ka future kharch (customer id se match hua)
feat["FutureSpend"] = feat["FutureSpend"].fillna(0)  # jo customer dobara nahi kharide unhe 0 mila
feat["WillBuy"] = (feat["FutureSpend"] > 0).astype(int)  # 1 = dobara kharida, 0 = nahi kharida
feat.shape  # customers x columns

In [ ]:
feat.head()  # customer table ki jhalak

In [ ]:
feat["WillBuy"].value_counts(normalize=True)  # kitne customers ne dobara kharida (1) aur kitno ne nahi (0), hissa

## 7. Part A – Classification: Will the Customer Buy Again in 90 Days?
**Target:** `WillBuy` (1 = yes, 0 = no).

**Steps:**
1. Take the log of very skewed columns (a few huge customers should not confuse the model).
2. Split 80% train / 20% test (same class ratio in both).
3. Scale the columns – the scaler learns **only from train data** (no leakage).

In [ ]:
FEATURES = ["Recency", "Frequency", "Monetary", "TotalItems", "UniqueProducts", "CustomerAgeDays", "AvgOrderValue", "Spend_Last90", "IsUK"]  # saare models ke input columns
SKEWED = ["Frequency", "Monetary", "TotalItems", "UniqueProducts", "AvgOrderValue", "Spend_Last90"]  # paise aur ginti wale columns jo bahut skewed hain

In [ ]:
X = feat[FEATURES].copy()  # input table
X[SKEWED] = np.log1p(X[SKEWED])  # log(1+x) bahut bade values ka asar kam karta hai
y_class = feat["WillBuy"]  # classification ka target

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y_class, test_size=0.2, random_state=42, stratify=y_class)  # 80/20 split, class ka ratio same rakhte hue
print(X_train.shape, X_test.shape)  # dono hisson ka size

In [ ]:
scaler = StandardScaler()  # scaler banaya
X_train_s = scaler.fit_transform(X_train)  # scaling train data se seekhi aur lagayi
X_test_s = scaler.transform(X_test)  # wahi scaling test data par lagayi

### Create and train the models 
Every model is **trained on the train data**. After that we measure each model on **both train data and test data**.

In [ ]:
model_lr = LogisticRegression(max_iter=1000)  # logistic regression
model_knn = KNeighborsClassifier(n_neighbors=15)  # KNN jo 15 sabse paas ke customers dekhta hai
model_nb = GaussianNB()  # Gaussian Naive Bayes
model_dt = DecisionTreeClassifier(max_depth=5, random_state=42)  # decision tree, depth limit ki taaki overfitting na ho
model_rf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42)  # 200 trees ka random forest

In [ ]:
model_lr.fit(X_train_s, y_train)  # logistic regression ko train data par train kiya
model_knn.fit(X_train_s, y_train)  # KNN ko train data par train kiya
model_nb.fit(X_train_s, y_train)  # Gaussian Naive Bayes ko train data par train kiya
model_dt.fit(X_train_s, y_train)  # decision tree ko train data par train kiya
model_rf.fit(X_train_s, y_train)  # random forest ko train data par train kiya

### Helper function to score any model on any data
One function, used for every model and for both train and test data → short code, easy to explain.
- **Accuracy** – how many predictions are right overall
- **Precision** – of the customers we said "will buy", how many really bought
- **Recall** – of the customers who really bought, how many we caught
- **F1** – balance of precision and recall

In [ ]:
def get_clf_scores(model, X, y, name):  # function jo kisi bhi data par model ke scores deta hai
    pred = model.predict(X)  # model se predictions li
    return {  # saare scores ek dictionary mein
        "Model": name,  # model ka naam
        "Accuracy": accuracy_score(y, pred),  # kul kitni predictions sahi hain
        "Precision": precision_score(y, pred, zero_division=0),  # "will buy" predictions kitni sahi hain
        "Recall": recall_score(y, pred, zero_division=0),  # kitne asli buyers pakde
        "F1": f1_score(y, pred, zero_division=0),  # precision aur recall ka balance
    }  # dictionary khatam

### Scores on TRAIN data and on TEST data
Same function, only the data changes. Compare both tables to check **overfitting**.

In [ ]:
results_train = pd.DataFrame([  # train data ke scores, har model ki ek row
    get_clf_scores(model_lr, X_train_s, y_train, "Logistic Regression"),  # logistic regression ke train scores
    get_clf_scores(model_knn, X_train_s, y_train, "KNN"),  # KNN ke train scores
    get_clf_scores(model_nb, X_train_s, y_train, "Gaussian NB"),  # Gaussian Naive Bayes ke train scores
    get_clf_scores(model_dt, X_train_s, y_train, "Decision Tree"),  # decision tree ke train scores
    get_clf_scores(model_rf, X_train_s, y_train, "Random Forest"),  # random forest ke train scores
])  # train table khatam

In [ ]:
results_test = pd.DataFrame([  # test data ke scores, har model ki ek row
    get_clf_scores(model_lr, X_test_s, y_test, "Logistic Regression"),  # logistic regression ke test scores
    get_clf_scores(model_knn, X_test_s, y_test, "KNN"),  # KNN ke test scores
    get_clf_scores(model_nb, X_test_s, y_test, "Gaussian NB"),  # Gaussian Naive Bayes ke test scores
    get_clf_scores(model_dt, X_test_s, y_test, "Decision Tree"),  # decision tree ke test scores
    get_clf_scores(model_rf, X_test_s, y_test, "Random Forest"),  # random forest ke test scores
])  # test table khatam

In [ ]:
results_train.round(3)  # TRAIN data ke scores dikhao

In [ ]:
results_test.round(3).sort_values("F1", ascending=False)  # TEST data ke scores, sabse achha model upar

### 🔍 Overfitting check: Train minus Test
If a number is **much bigger** on train than on test (big positive gap), the model has memorised the train data = **overfitting**.

In [ ]:
gap_clf = results_train.set_index("Model") - results_test.set_index("Model")  # train score minus test score
gap_clf.round(3)  # gap dikhao (bada positive gap = overfitting)

### Baseline to beat (simple pandas)
If we just guess the **most common class** for every customer, how accurate are we? A real model must beat this number, otherwise it is useless.

In [ ]:
baseline_acc = y_test.value_counts(normalize=True).max()  # sabse common class ka hissa = hamesha wahi guess karne par accuracy
print("Baseline accuracy:", round(baseline_acc, 3))  # baseline accuracy print karo

### Pick the best model
We choose by **F1 on the test data** (balance of precision and recall).

In [ ]:
models_clf = {"Logistic Regression": model_lr, "KNN": model_knn, "Gaussian NB": model_nb, "Decision Tree": model_dt, "Random Forest": model_rf}  # naam -> trained model
best_name = results_test.sort_values("F1", ascending=False).iloc[0]["Model"]  # F1 ke hisaab se sabse achhe model ka naam
best_model = models_clf[best_name]  # sabse achha trained model
pred_best = best_model.predict(X_test_s)  # test data par uski predictions
print(best_name)  # winner dikhao

### 📊 Chart 2 – Confusion Matrix of the Best Model
- **Top-left (TN)** – said "will not buy", really did not buy ✅
- **Bottom-right (TP)** – said "will buy", really bought ✅
- **Top-right (FP)** – said "will buy", but did not buy ❌ (wasted offer)
- **Bottom-left (FN)** – said "will not buy", but bought ❌ (missed customer)

In [ ]:
def plot_confusion(y_true, y_pred, title):  # function jo confusion matrix banata hai
    cm = confusion_matrix(y_true, y_pred)  # sahi aur galat predictions ginti
    plt.figure(figsize=(4.5, 3.5))  # chart ka size
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Will not buy", "Will buy"], yticklabels=["Will not buy", "Will buy"])  # numbers ke saath rangin grid
    plt.xlabel("Predicted")  # x axis ka label
    plt.ylabel("Actual")  # y axis ka label
    plt.title(title)  # chart ka title
    plt.show()  # chart dikhao

In [ ]:
plot_confusion(y_test, pred_best, f"Confusion Matrix - {best_name}")  # function call karke chart 2 banao

In [ ]:
tn, fp, fn, tp = confusion_matrix(y_test, pred_best).ravel()  # matrix ko uske chaar numbers mein tod diya
print("TN", tn, "FP", fp, "FN", fn, "TP", tp)  # chaaron numbers print karo

### 🔍 Error analysis – where does the model go wrong?
We compare the average behaviour of customers predicted **correctly** vs **wrongly**.

In [ ]:
test_view = feat.loc[X_test.index, ["Recency", "Frequency", "Monetary"]].copy()  # test customers ki asli (bina log wali) values
test_view["Wrong"] = pred_best != y_test.values  # True jahan model ne galti ki
test_view.groupby("Wrong").mean().round(1)  # sahi vs galat predictions ke liye average Recency, Frequency, Monetary

### 🔑 What drives repeat buying? (Random Forest feature importance)

In [ ]:
imp = pd.Series(model_rf.feature_importances_, index=FEATURES)  # har input column ki importance
imp.sort_values(ascending=False).round(3)  # sabse important columns pehle dikhao

### 💰 Business impact
Customers the model says **will NOT come back** are the ones for a retention offer. How much money did they bring before?

In [ ]:
test_view["Predicted"] = pred_best  # table mein model ki prediction jodi
at_risk = test_view[test_view["Predicted"] == 0]  # wo customers jinke liye predict kiya ki wapas NAHI aayenge
share_at_risk = at_risk["Monetary"].sum() / test_view["Monetary"].sum()  # test group mein unka past kharch ka hissa
print(len(at_risk), f"customers flagged, they brought {share_at_risk:.1%} of past spend")  # business impact print karo

## 8. Part B – Regression: How Much Will a Returning Customer Spend?
**Target:** `FutureSpend` in the next 90 days, only for customers who **did buy** (so the target is above zero).

Spend is very skewed (a few big buyers), so we train on `log(1 + spend)` and **convert predictions back to £** before measuring errors.
- **MAE** – average error in £
- **MSE / RMSE** – like MAE but big mistakes are punished more (RMSE is in £)
- **MAPE** – average error in % (can look big because of very small orders)
- **R²** – how much of the ups and downs the model explains

In [ ]:
reg_data = feat[feat["WillBuy"] == 1]  # sirf wo customers jinhone agle 90 din mein kharida
X_reg = X.loc[reg_data.index]  # unke input features (log pehle hi ho chuka)
y_reg = np.log1p(reg_data["FutureSpend"])  # future kharch ka log, kyunki kharch bahut skewed hai

In [ ]:
Xr_train, Xr_test, yr_train, yr_test = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)  # 80/20 split
scaler_r = StandardScaler()  # regression data ke liye naya scaler
Xr_train_s = scaler_r.fit_transform(Xr_train)  # scaling train data se seekhi aur lagayi
Xr_test_s = scaler_r.transform(Xr_test)  # wahi scaling test data par lagayi

In [ ]:
reg_lr = LinearRegression()  # linear regression
reg_knn = KNeighborsRegressor(n_neighbors=15)  # KNN regressor
reg_dt = DecisionTreeRegressor(max_depth=5, random_state=42)  # decision tree regressor
reg_rf = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42)  # random forest regressor

In [ ]:
reg_lr.fit(Xr_train_s, yr_train)  # linear regression ko train data par train kiya
reg_knn.fit(Xr_train_s, yr_train)  # KNN ko train data par train kiya
reg_dt.fit(Xr_train_s, yr_train)  # decision tree ko train data par train kiya
reg_rf.fit(Xr_train_s, yr_train)  # random forest ko train data par train kiya

In [ ]:
def get_reg_scores(model, X, y_log, name):  # function jo kisi bhi data par model ke errors £ mein deta hai
    pred = np.expm1(model.predict(X))  # predictions ko wapas £ mein badla
    actual = np.expm1(y_log)  # asli values ko wapas £ mein badla
    mse = mean_squared_error(actual, pred)  # mean squared error
    return {  # saare errors ek dictionary mein
        "Model": name,  # model ka naam
        "MAE": mean_absolute_error(actual, pred),  # £ mein average error
        "MSE": mse,  # mean squared error
        "RMSE": np.sqrt(mse),  # MSE ka square root (£ mein)
        "MAPE %": mean_absolute_percentage_error(actual, pred) * 100,  # percent mein average error
        "R2": r2_score(actual, pred),  # model kitna ups-downs samjha pata hai
    }  # dictionary khatam

### Errors on TRAIN data and on TEST data
Same function, only the data changes.

In [ ]:
reg_train = pd.DataFrame([  # train data ke errors, har model ki ek row
    get_reg_scores(reg_lr, Xr_train_s, yr_train, "Linear Regression"),  # linear regression ke train errors
    get_reg_scores(reg_knn, Xr_train_s, yr_train, "KNN"),  # KNN ke train errors
    get_reg_scores(reg_dt, Xr_train_s, yr_train, "Decision Tree"),  # decision tree ke train errors
    get_reg_scores(reg_rf, Xr_train_s, yr_train, "Random Forest"),  # random forest ke train errors
])  # train table khatam

In [ ]:
results_reg_test = pd.DataFrame([  # test data ke errors, har model ki ek row
    get_reg_scores(reg_lr, Xr_test_s, yr_test, "Linear Regression"),  # linear regression ke test errors
    get_reg_scores(reg_knn, Xr_test_s, yr_test, "KNN"),  # KNN ke test errors
    get_reg_scores(reg_dt, Xr_test_s, yr_test, "Decision Tree"),  # decision tree ke test errors
    get_reg_scores(reg_rf, Xr_test_s, yr_test, "Random Forest"),  # random forest ke test errors
])  # test table khatam

In [ ]:
reg_train.round(2).sort_values("MAE")  # TRAIN data ke errors dikhao

In [ ]:
results_reg_test.round(2).sort_values("MAE")  # TEST data ke errors, sabse kam MAE (best) upar

### 🔍 Overfitting check: Test minus Train
For errors (MAE, RMSE ...) a **big positive number here** means the test error is much worse than the train error = **overfitting**. For R2 a big negative number means the same.

In [ ]:
gap_reg = results_reg_test.set_index("Model") - reg_train.set_index("Model")  # test error minus train error
gap_reg.round(2)  # gap dikhao (bada positive gap = overfitting)

### Baseline to beat (simple pandas)
If we guess the **average spend** for every customer, what is the average error (MAE)? A real model must have a lower MAE than this.

In [ ]:
actual_spend = np.expm1(yr_test)  # test customers ka asli future kharch £ mein
baseline_mae = (actual_spend - actual_spend.mean()).abs().mean()  # hamesha average kharch guess karein to average error
print("Baseline MAE: £", round(baseline_mae))  # baseline error print karo

## 9. Part C – Customer Segmentation (Clustering)
Here there is **no target** – the algorithm finds groups on its own using **Recency, Frequency, Monetary**.
We log-transform and scale them first, because K-Means and hierarchical clustering work with distances.

In [ ]:
rfm = np.log1p(feat[["Recency", "Frequency", "Monetary"]])  # RFM columns ka log, skew kam karne ke liye
X_rfm = StandardScaler().fit_transform(rfm)  # scale kiya taaki har column ka weight barabar ho

### How many clusters? (Elbow method)
We try k = 2 to 8 and note the **inertia** (lower = tighter groups). The point where the line bends like an elbow is a good k.
`.apply()` runs our small function for every k, so no loop is needed.

In [ ]:
def get_inertia(k):  # function jo ek k ke liye K-Means ka inertia deta hai
    km = KMeans(n_clusters=k, random_state=42, n_init=10)  # k clusters wala K-Means
    km.fit(X_rfm)  # RFM data par K-Means fit kiya
    return km.inertia_  # clusters kitne tight hain (kam = zyada tight)

In [ ]:
ks = pd.Series(range(2, 9))  # k ki values jo try karni hain: 2 se 8
inertias = ks.apply(get_inertia)  # har k ke liye function chalaya
inertias.round(0)  # inertia ki values dikhao

### 📉 Chart 3 – Elbow Chart

In [ ]:
def plot_elbow(ks, inertias):  # function jo elbow chart banata hai
    plt.figure(figsize=(6, 4))  # chart ka size
    sns.lineplot(x=ks, y=inertias, marker="o")  # clusters ki ginti ke saamne inertia
    plt.title("Elbow Chart - K-Means")  # chart ka title
    plt.xlabel("Number of clusters (k)")  # x axis ka label
    plt.ylabel("Inertia (lower = tighter groups)")  # y axis ka label
    plt.show()  # chart dikhao

In [ ]:
plot_elbow(ks, inertias)  # function call karke chart 3 banao

We choose **k = 4** because it is easy to explain to a business and the elbow shows it is a reasonable choice.

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)  # 4 segments wala final K-Means
feat["Cluster"] = kmeans.fit_predict(X_rfm)  # har customer ko segment number diya
print(feat["Cluster"].value_counts())  # har segment mein customers ki ginti
print("Silhouette:", round(silhouette_score(X_rfm, feat["Cluster"], sample_size=3000, random_state=42), 3))  # segments kitne alag hain (zyada = behtar)

### Segment profile
Average behaviour of each segment. The last two columns come from the **future** data, so they show whether the segments really differ in value.

🗣️ After reading the table, give each segment a business name yourself (e.g. *VIP*, *Loyal*, *Sleeping*, *Low value*) – low Recency + high Frequency + high Monetary = VIP.

In [ ]:
columns = ["Recency", "Frequency", "Monetary", "WillBuy", "FutureSpend"]  # wo columns jinka average chahiye
segments = feat.groupby("Cluster")[columns].mean().round(2)  # har segment ke liye in columns ka average
segments["Customers"] = feat["Cluster"].value_counts()  # har segment mein customers ki ginti jodi
segments.sort_values("Monetary", ascending=False)  # sabse zyada kharch karne wala segment upar dikhao

### Hierarchical clustering vs K-Means
Hierarchical clustering is slow on big data, so we compare both on a **random half of the customers**.
`adjusted_rand_score` tells how much the two methods agree (1 = same groups, 0 = random).

In [ ]:
sample = pd.DataFrame(X_rfm).sample(frac=0.5, random_state=42)  # customers ka random 50%
hier = AgglomerativeClustering(n_clusters=4, linkage="ward")  # ward linkage wala hierarchical clustering
hier_labels = hier.fit_predict(sample)  # sample ke hierarchical cluster labels
km_labels = kmeans.predict(sample.values)  # wahi customers ke K-Means labels

In [ ]:
print("Hierarchical silhouette:", round(silhouette_score(sample, hier_labels), 3))  # hierarchical clusters ki quality
print("K-Means silhouette:", round(silhouette_score(sample, km_labels), 3))  # wahi customers par K-Means clusters ki quality
print("Agreement (adjusted rand):", round(adjusted_rand_score(hier_labels, km_labels), 3))  # dono methods kitna sahmat hain

## 10. Final Numbers (printed automatically)

In [ ]:
clf_scores = results_test.set_index("Model")  # classification table jisme model ka naam row label hai
print("Best classifier:", best_name)  # sabse achhe classifier ka naam
print("Its F1:", round(clf_scores.loc[best_name, "F1"], 3), "| Recall:", round(clf_scores.loc[best_name, "Recall"], 3))  # uska F1 aur Recall
print("Its accuracy:", round(clf_scores.loc[best_name, "Accuracy"], 3), "| Baseline accuracy:", round(baseline_acc, 3))  # accuracy vs baseline accuracy

In [ ]:
reg_scores = results_reg_test.set_index("Model")  # regression table jisme model ka naam row label hai
best_reg = reg_scores["MAE"].idxmin()  # sabse kam MAE wala regressor
print("Best regressor:", best_reg)  # sabse achhe regressor ka naam
print("Its MAE: £", round(reg_scores.loc[best_reg, "MAE"]), "| Baseline MAE: £", round(baseline_mae))  # MAE vs baseline MAE
print(f"Top 20% customers give {top20:.1%} of revenue")  # Pareto ka fact

## 11. Final Conclusion

**What we built (end to end):** cleaned ~1M invoice lines, made one row per customer from past data, and solved three business problems – *segment*, *predict repeat purchase*, *forecast spend*.

**Key results (see the numbers printed above):**
- **Classification:** compare the best model's accuracy with the baseline accuracy printed above, and look at its F1 / Recall. The feature-importance table shows which behaviours (e.g. Recency, recent spend) matter most for repeat buying.
- **Regression:** compare the best model's MAE with the baseline MAE – a lower MAE means it beats simply guessing the average spend.
- **Segmentation:** K-Means gave 4 groups; the adjusted-rand score shows how much hierarchical clustering agrees. Check the `WillBuy` and `FutureSpend` columns of the segment table – if they differ a lot between segments, the segments are useful in real life.
- **Business insight:** see the Pareto number printed above – the top 20% customers' share of revenue shows how important it is to protect them.

**How the company can use it:**
1. Send loyalty offers to the highest-value segments.
2. Send a win-back discount to customers predicted **not to buy**.
3. Plan stock and marketing budget using the **spend forecast**.

**Limitations (be honest in the interview):**
- Only past behaviour is used – no product category, price discounts, or marketing data.
- One cutoff date was used; a rolling cutoff would be more robust.
- MAPE is high for tiny orders; MAE in £ is the better error here.
- Models used default-style settings with light tuning.

**Next steps:** try hyper-parameter tuning (GridSearchCV), add product-category features, test several cutoff dates, and build a small dashboard for the marketing team.